## 1 概述

为解决当序列较长时，RNN 反向传播的梯度连乘会导致梯度消失的问题，长短期记忆网络（Long Short-Term Memory，LSTM） 应运而生。

它的核心思想是：将“记忆”与“输出”解耦。RNN 中，$H_t$ 既当记忆又当输出，导致两者互相牵制；LSTM 则引入独立的细胞状态（Cell State），作为纯粹的“记忆载体”，并通过门控机制（Gating Mechanism）——遗忘门、输入门、输出门——让网络自主决定“记住什么、更新什么、输出什么”。

## 2 门控机制

|  | 计算公式 | 线性部分 |
|---|---|---|
| 遗忘门（Forget Gate） | $f_t = \sigma(Z_f)$ | $Z_f=X_t W_{xf} + H_{t-1} W_{hf} + b_f$ |
| 输入门（Input Gate） | $i_t = \sigma(Z_i)$ | $Z_i=X_t W_{xi} + H_{t-1} W_{hi} + b_i$ |
| 输出门（Output Gate） | $o_t = \sigma(Z_o)$ | $Z_o=X_t W_{xo} + H_{t-1} W_{ho} + b_o$ |
| 候选记忆（Cell State） | $\tilde{C}_t = \tanh(Z_c)$ | $Z_c=X_t W_{xc} + H_{t-1} W_{hc} + b_c$ |
| 更新记忆（Cell State） | $C_t = f_t\odot C_{t-1}+i_t\odot \tilde{C}_t$ |  |
| 隐藏状态（Hidden State） | $H_t = o_t\odot \tanh(C_t)$ |  |

看起来式子很多很复杂，其实很有规律：
- 前四个式子结构相同（线性变换+激活函数），各自包含两个权重和一个偏置，它们各自独立，但都是一样进行随机初始化。
- $\sigma$ 就是 $\text{Sigmoid}$ 在公式中的简写，它输出 $(0,1)$ 的连续值，它与矩阵逐元素相乘可以做到**按比例控制矩阵中信息的保留或丢弃**。

|  | 作用 | 举例 | 理解 |
|-|-|-|-|
| $f_t\odot C_{t-1}$ | 控制旧记忆的保留或丢弃 | 若 $f_t=0.7$ 则表示保留 70% 丢弃 30% | 要遗忘多少旧记忆 |
| $i_t\odot \tilde{C}_{t}$ | 控制新记忆的保留或丢弃 | 若 $i_t=0.7$ 则表示保留 70% 丢弃 30% | 要记住多少新记忆 |
| $o_t\odot \tanh(C_t)$ | 控制新记忆有多少传递下去 | 若 $o_t=0.7$ 则表示传递 70% 丢弃 30% | 要说出多少新记忆 <br> $C_t$ 由加法得来，取值范围不限于 $(-1,1)$ ，故用 $\tanh$ |

## 3 反向传播

门控机制怎么就能解决 RNN 的梯度消失呢？

原理是：把 RNN 中既当记忆又当输出的 $H_t$ 拆分出 $C_t$ 当记忆，$H_t$ 只当输出。代价 $J$ 当然还是依赖 $H_T$ ，但只限于空间层上用各层的 $H_T^{[l]}$ 进行传播；而在时间步上传播时不再用 $H_t$ 而是用 $C_t$ ，由于 $C_t$ 是通过加法更新的，几乎不会衰减，这就避免了 $\tanh^{'}$ 连乘导致的梯度消失问题。

一句话的原理，其实内部还有不少细节，首先看整体的流转结构：

| 传播维度 | 承载对象 | 误差信号 | 目的 | 备注 |
| - | - | - | - | - |
| 空间层 | $H_T^{[l]}$ | $\delta_T^{[l]}=\frac{\partial J}{\partial H_T^{[l]}}$ | 把误差从上层传到下层 |  |
| 时间步 | $C_t$ | $\delta_t=\frac{\partial J}{\partial C_t}$ | 把误差从后一刻传到前一刻 | 不影响空间层传播 |
| 时间步（分支） | $Z_f \\ Z_i \\ Z_o \\ Z_c$ | $\delta_f=\frac{\partial J}{\partial Z_f} \\ \delta_i=\frac{\partial J}{\partial Z_i} \\ \delta_o=\frac{\partial J}{\partial Z_o} \\ \delta_c=\frac{\partial J}{\partial Z_c}$ | 计算梯度，更新权重和偏置 | 不参与传播，只更新参数 |

在 RNN 中就已了解，时间步传播时的误差是递推式子，LSTM 也是一样：
$$
\delta_{t-1}=\frac{\partial J}{\partial C_{t-1}}=\frac{\partial J}{\partial C_t} \cdot \frac{\partial C_t}{\partial C_{t-1}}
$$

已知 $\delta_t=\frac{\partial J}{\partial C_t}$ ，接着看：
$$
\frac{\partial C_t}{\partial C_{t-1}}=(f_t\odot C_{t-1}+i_t\odot \tilde{C}_t)_{C_{t-1}}^{'}=f_t
$$

因此：
$$
\delta_{t-1}=\delta_t \odot f_t
$$

还记得 RNN 中的递推式子是：$\delta_{t-1}=(\delta_t \cdot W_{hh}^T)\odot \tanh^{'}(Z_{t-1})$ ，而 LSTM 这里没有了 $\tanh^{'}$ 的连乘，遗忘门 $f_t$ 是 Sigmoid 输出，取值在 $(0,1)$ 之间，且是网络自己学习的，可以通过训练让 $f_t$ 逼近 1 ，从而让梯度几乎无损地在序列中传播，长序列的梯度消失问题由此而解决。

至此，还有个细节问题，误差在更改承载对象时是如何换算的呢？要推导这个需要先了解雅可比矩阵。

## 4 雅可比矩阵

通俗来说，雅可比矩阵（Jacobian Matrix）就是“多变量、多输出”情况下的导数矩阵。

假设有式子 $Y=f(X)$ ，若是“多变量、单输出”的场景，即 $X=(x_1,x_2,...,x_n)\,,\,Y=y$ ，那么其导数是一个向量：
$$
\frac{\partial Y}{\partial X}=\left[\frac{\partial y}{\partial x_1},\frac{\partial y}{\partial x_2},...,\frac{\partial y}{\partial x_n}\right]
$$

但若是“多变量、多输出”的场景，即 $X=(x_1,x_2,...,x_n)\,,\,Y=(y_1,y_2,...,y_m)$ ，那么其导数是一个矩阵，即**雅可比矩阵**：
$$
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \frac{\partial y_1}{\partial x_2} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\frac{\partial y_2}{\partial x_1} & \frac{\partial y_2}{\partial x_2} & \cdots & \frac{\partial y_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \frac{\partial y_m}{\partial x_2} & \cdots & \frac{\partial y_m}{\partial x_n} \\
\end{pmatrix}
$$

在神经网络中，雅可比矩阵无处不在，激活函数场景通常都是 $n=m$ ，全连接场景却不然。

这里为了简化推导，我们假设输入输出维度相同的场景 $X=[x_1,x_2,...,x_n]\,,\,Y=[y_1,y_2,...,y_n]$ ，先看全连接的 $Y=XW$ ，此时：
$$
y_1=x_1w_{11}+x_2w_{21}+\cdots+x_nw_{n1} \\ 
y_2=x_1w_{12}+x_2w_{22}+\cdots+x_nw_{n2} \\ 
\vdots \\
y_n=x_1w_{1n}+x_2w_{2n}+\cdots+x_nw_{nn} \\ 
$$
会发现，每一个 $y$ 都依赖所有的 $x$ ，这就是全连接；反向传播求得导数 $\frac{\partial Y}{\partial X}=W^T$ 是一个稠密的雅可比矩阵（所有位置都有值），此时误差的计算是矩阵乘法 $\delta_X=\delta_Y \cdot W^T$。

但如果是 LR/RNN/LSTM 中的激活函数（Sigmoid/ReLU/tanh），以 tanh 为例：
$$
y_1=\tanh(x_1) \\
y_2=\tanh(x_2) \\
\vdots \\
y_n=\tanh(x_n)
$$
会发现，每一个 $y$ 都仅仅依赖与之相同下标的 $x$ ；当反向传播求导数时：
- 若下标相等，$\frac{\partial y_i}{\partial x_i}=\tanh^{'}(x_i)=1-\tanh^2(x_i)$
- 若下标不等 $i \ne j$，由于 $y_j$ 与 $x_i$ 无关，因此 $\frac{\partial y_j}{\partial x_i}=0$

所以得到的雅可比矩阵就是一个对角矩阵（非主对角线都是零），如下：
$$
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & 0 & \cdots & 0 \\
0 & \frac{\partial y_2}{\partial x_2} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{\partial y_n}{\partial x_n} \\
\end{pmatrix}
$$

我们用 $D$ 表示这个雅可比矩阵，在神经网络中，误差信号的传递通常是这样 $\delta_{new}=\delta \cdot D$ ，其中：
- 误差向量 $\delta$ 是形状 $1 \times n$ 的向量 $[a_1,a_2,\cdots,a_n]$
- 雅可比矩阵 $D$ 是形状 $n \times n$ 的对角阵，主对角线是 $d_1,d_2,\cdots,d_n$

那么：
$$
\begin{pmatrix}
a_1 & a_2 & \cdots & a_n 
\end{pmatrix} \cdot \begin{pmatrix}
d_1 & 0 & \cdots & 0 \\
0 & d_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & d_n \\
\end{pmatrix} = \begin{pmatrix} a_1d_1 & a_2d_2 & \cdots & a_nd_n \end{pmatrix}
$$
显然，结果其实就是逐元素相乘 $\odot$ （哈达玛积），因此在实践中不会去构建对角阵 $D$ ，也不会用矩阵乘法，而是将它写成 $1 \times n$ 的向量 $D=[d1,d2,\cdots,d_n]$ ，用逐元素相乘：
$$
\delta_{new}=\delta \odot D=\begin{pmatrix}
a_1 & a_2 & \cdots & a_n 
\end{pmatrix} \odot \begin{pmatrix}
d_1 & d_2 & \cdots & d_n 
\end{pmatrix}
$$

这就是为什么公式推导中常见，上一步还是矩阵乘法 $\cdot$ ，下一步就变成了逐元素相乘 $\odot$ 。

## 5 误差换算

至此，还有个细节问题，误差在更改承载对象时是如何换算的呢？先给出结果：

|  |换算公式|
| - | - |
| 从 $H_T$ 到 $C_t$ | $\delta_{C_t}=\delta_{H_t} \odot o_t \odot \tanh^{'}(C_t) + \delta_{C_{t+1}} \odot f_{t+1}$ |
| 从 $H_t$ 到 $Z_o$ | $\delta_{Z_o}=\delta_{H_t} \odot \tanh(C_t) \odot \sigma^{'}(Z_o)$ |
| 从 $C_t$ 到 $Z_f$ | $\delta_{Z_f}=\delta_{C_t} \odot C_{t-1} \odot \sigma^{'}(Z_f)$ |
| 从 $C_t$ 到 $Z_i$ | $\delta_{Z_i}=\delta_{C_t} \odot \tilde{C}_t \odot \sigma^{'}(Z_i)$ |
| 从 $C_t$ 到 $Z_c$ | $\delta_{Z_c}=\delta_{C_t} \odot i_t \odot (1-\tilde{C}_t^2)$ |

**从 $H_T$ 到 $C_t$** ，已知 $H_t$ 与 $C_t$ 的关系：$H_t=o_t \odot \tanh(C_t)$ ，因此：
$$
\frac{\partial J}{\partial C_t}=\frac{\partial J}{\partial H_t} \cdot \frac{\partial H_t}{\partial C_t}=\delta_{H_t} \odot o_t \odot \tanh^{'}(C_t)
$$

这是当前时刻的误差换算，还要加上后一时刻（因为是反向传播）传递过来的误差，所以最后的式子是：
$$
\delta_{C_t}=\frac{\partial J}{\partial C_t}=\delta_{H_t} \odot o_t \odot \tanh^{'}(C_t) + \delta_{C_{t+1}} \odot f_{t+1}
$$

**从 $H_T$ 到 $Z_o$** ，输出门与三个分支不同，因为 $o_t$ 不在 $C_t$ 的式子中，而是直接参与 $H_t$ 的更新，其传导途径是 $H_t \to o_t \to Z_o$ ，因此：
$$
\delta_{Z_o} = \frac{\partial J}{\partial Z_o} = \frac{\partial J}{\partial H_t} \cdot \frac{\partial H_t}{\partial o_t} \cdot \frac{\partial o_t}{\partial Z_o} = \delta_{H_t} \odot \tanh(C_t) \odot \sigma^{'}(Z_o)
$$

**从 $C_t$ 到 $Z_f$** ，传导途径是 $C_t \to f_t \to Z_f$ ，因此：
$$
\delta_{Z_f} = \frac{\partial J}{\partial Z_t} = \frac{\partial J}{\partial C_t} \cdot \frac{\partial C_t}{\partial f_t} \cdot \frac{\partial f_t}{\partial Z_t} = \delta_{C_t} \odot C_{t-1} \odot \sigma^{'}(Z_f)
$$

**从 $C_t$ 到 $Z_i$** ，传导途径是 $C_t \to i_t \to Z_i$ ，因此：
$$
\delta_{Z_i} = \frac{\partial J}{\partial Z_i} = \frac{\partial J}{\partial C_t} \cdot \frac{\partial C_t}{\partial i_t} \cdot \frac{\partial i_t}{\partial Z_i} = \delta_{C_t} \odot \tilde{C}_{t} \odot \sigma^{'}(Z_i)
$$

**从 $C_t$ 到 $Z_c$** ，传导途径是 $C_t \to \tilde{C}_t \to Z_c$ ，因此：
$$
\delta_{Z_c} = \frac{\partial J}{\partial Z_c} = \frac{\partial J}{\partial C_t} \cdot \frac{\partial C_t}{\partial \tilde{C}_t} \cdot \frac{\partial \tilde{C}_t}{\partial Z_c} = \delta_{C_t} \odot i_t \odot (1-\tilde{C}_t^2)
$$

